In [1]:
"""
Step 8: Clinical Utility Analysis - EXACT MATCH TO PAPER
Replicates the exact methodology from Yu et al.'s GitHub code
Key additions:
- Savitzky-Golay smoothing filter
- Finer threshold granularity (0.005 steps)
- Min benefit clipping (-0.2)
- All 10 models
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter  # ← PAPER USES THIS
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('.')


In [2]:
def decision_curve_analysis(y_true, y_pred_proba, threshold_range=None):
    """
    Calculate net benefit for decision curve analysis.
    Exactly as in paper's utils.py
    """
    if threshold_range is None:
        threshold_range = np.arange(0.01, 0.99, 0.005)  # ← PAPER USES 0.005
    
    from sklearn.metrics import confusion_matrix
    
    prevalence = np.mean(y_true)
    
    net_benefit_model = []
    net_benefit_all = []
    net_benefit_none = []
    
    for threshold in threshold_range:
        y_pred = (y_pred_proba >= threshold).astype(int)
        
        if len(np.unique(y_pred)) == 1:
            if y_pred[0] == 1:
                tp = np.sum(y_true == 1)
                fp = np.sum(y_true == 0)
                tn = fn = 0
            else:
                tn = np.sum(y_true == 0)
                fn = np.sum(y_true == 1)
                tp = fp = 0
        else:
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        n = len(y_true)
        
        if (tp + fp) > 0:
            nb_model = (tp/n) - (fp/n) * (threshold/(1-threshold))
        else:
            nb_model = 0
        
        nb_all = prevalence - (1-prevalence) * (threshold/(1-threshold))
        nb_none = 0
        
        net_benefit_model.append(nb_model)
        net_benefit_all.append(nb_all)
        net_benefit_none.append(nb_none)
    
    return threshold_range, net_benefit_model, net_benefit_all, net_benefit_none


In [3]:
def load_model_and_data(model_name):
    """Load trained model and test data - OPTIMIZED FOR SPEED"""
    print(f"\nLoading {model_name}...")
    
    test = pd.read_csv("../results/data/test_data.csv")
    exclude_cols = ['subject_id', 'hadm_id', 'stay_id', 'mortality_28day']
    feature_cols = [col for col in test.columns if col not in exclude_cols]
    
    X_test = test[feature_cols]
    y_test = test['mortality_28day']
    
    train = pd.read_csv("../results/data/train_data.csv")
    X_train_full = train[feature_cols]
    y_train_full = train['mortality_28day']
    
    from sklearn.model_selection import train_test_split
    
    # Use smaller sample for slow models
    if model_name in ["SVM", "MLP", "Neural Network", "AdaBoost"]:
        print(f"  Using 30% training sample for speed")
        X_train_sample, _, y_train_sample, _ = train_test_split(
            X_train_full, y_train_full, train_size=0.3, random_state=42, stratify=y_train_full
        )
        X_train, X_cal, y_train, y_cal = train_test_split(
            X_train_sample, y_train_sample, test_size=0.2, random_state=42, stratify=y_train_sample
        )
    else:
        X_train, X_cal, y_train, y_cal = train_test_split(
            X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
        )
    
    # Import models
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier
    from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB
    from sklearn.svm import LinearSVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.calibration import CalibratedClassifierCV
    
    param_file = f"../results/models/{model_name.lower().replace(' ', '_')}_params.csv"
    
    if os.path.exists(param_file):
        params_df = pd.read_csv(param_file)
        params = params_df.to_dict('records')[0]
    else:
        params = {}
    
    # Initialize models with speed optimizations
    if model_name == "LightGBM":
        base_model = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, **params)
    elif model_name == "XGBoost":
        base_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', **params)
    elif model_name == "Gradient Boosting":
        base_model = GradientBoostingClassifier(random_state=42, **params)
    elif model_name == "Random Forest":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    elif model_name == "Logistic Regression":
        base_model = LogisticRegression(random_state=42, max_iter=1000, **params)
    elif model_name == "Naive Bayes":
        base_model = GaussianNB(**params)
    elif model_name == "SVM":
        # LinearSVC for speed
        base_model = LinearSVC(random_state=42, max_iter=1000, dual='auto')
        base_model.fit(X_train, y_train)
        calibrated_model = CalibratedClassifierCV(base_model, method='sigmoid', cv='prefit')
        calibrated_model.fit(X_cal, y_cal)
        y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
        print(f"  Trained and calibrated")
        return calibrated_model, X_test, y_test, y_pred_proba
    elif model_name == "KNN":
        params_safe = {k: v for k, v in params.items() if k != 'n_neighbors'}
        params_safe['n_neighbors'] = min(params.get('n_neighbors', 20), 50)
        base_model = KNeighborsClassifier(n_jobs=-1, **params_safe)
    elif model_name == "Decision Tree":
        base_model = DecisionTreeClassifier(random_state=42, **params)
    elif model_name in ["Neural Network", "MLP"]:
        base_model = MLPClassifier(random_state=42, hidden_layer_sizes=(50,), 
                                   max_iter=100, early_stopping=True)
    elif model_name == "AdaBoost":
        base_model = AdaBoostClassifier(random_state=42, n_estimators=50, algorithm='SAMME')
    else:
        raise ValueError(f"Model {model_name} not supported")
    
    base_model.fit(X_train, y_train)
    
    calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv='prefit', n_jobs=-1)
    calibrated_model.fit(X_cal, y_cal)
    
    y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
    
    print(f"  Trained and calibrated")
    return calibrated_model, X_test, y_test, y_pred_proba


In [4]:
def create_decision_curve_PAPER_STYLE(models_data, output_dir):
    """
    Create decision curve EXACTLY like the paper's Figure 4
    Uses smoothing and proper formatting
    """
    print("\n" + "="*70)
    print("DECISION CURVE ANALYSIS - PAPER STYLE")
    print("="*70)
    
    plt.figure(figsize=(15, 8))
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # ← PAPER'S KEY SETTINGS
    thresholds = np.arange(0.01, 0.99, 0.005)  # Finer granularity
    min_benefit = -0.2  # Clip negative values
    
    # Paper's color scheme (approximated)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
              '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    # Plot reference strategies FIRST (as in paper)
    y_true_ref = list(models_data.values())[0]['y_true']
    y_pred_ref = list(models_data.values())[0]['y_pred_proba']
    _, _, nb_all, nb_none = decision_curve_analysis(y_true_ref, y_pred_ref, thresholds)
    
    plt.plot(thresholds, nb_all, 'k--', lw=2, label='Treat All')
    plt.plot(thresholds, nb_none, 'k:', lw=2, label='Treat None')
    
    # Plot each model WITH SMOOTHING (paper's key feature)
    for idx, (model_name, data) in enumerate(models_data.items()):
        y_true = data['y_true']
        y_pred_proba = data['y_pred_proba']
        
        # Calculate decision curve
        thresh, nb_model, _, _ = decision_curve_analysis(y_true, y_pred_proba, thresholds)
        
        # ← PAPER'S SMOOTHING: Clip then smooth
        nb_model_array = np.array(nb_model)
        nb_model_clipped = np.maximum(nb_model_array, min_benefit)
        
        # Apply Savitzky-Golay filter (makes curves look professional)
        nb_model_smooth = savgol_filter(
            nb_model_clipped,
            window_length=21,  # Paper's setting
            polyorder=3        # Paper's setting
        )
        
        plt.plot(thresholds, nb_model_smooth, 
                '-', color=colors[idx % len(colors)], 
                lw=2, label=model_name)
        
        # Calculate metrics
        mask = (thresholds >= 0.1) & (thresholds <= 0.5)
        auc_dca = np.trapz(nb_model_smooth[mask], thresholds[mask])
        
        print(f"\n{model_name}:")
        print(f"  AUC-DCA (0.1-0.5): {auc_dca:.4f}")
        print(f"  Max Net Benefit: {np.max(nb_model_smooth):.4f}")
    
    # Formatting (match paper)
    plt.xlim([0, 1])
    plt.ylim([min_benefit, 0.4])
    plt.xlabel('Threshold Probability', fontsize=12)
    plt.ylabel('Net Benefit', fontsize=12)
    plt.title('Decision Curve Analysis', fontsize=14)
    plt.legend(loc='upper right', fontsize=10)
    plt.tight_layout()
    
    # Save PNG first (prevents Windows Edge file association issue)
    plt.savefig(f'{output_dir}/decision_curve_analysis_all_models.png', 
               format='png', bbox_inches='tight', dpi=300)
    # Optionally save SVG for vector graphics (comment out if not needed)
    # plt.savefig(f'{output_dir}/decision_curve_analysis_all_models.svg', 
    #            format='svg', bbox_inches='tight', dpi=300)
    
    print(f"\n  Saved: {output_dir}/decision_curve_analysis_all_models.png")
    plt.close()


In [5]:
def create_calibration_curves(models_data, output_dir, top_n=3):
    """Create calibration curves for top N models"""
    print("\n" + "="*70)
    print(f"CALIBRATION CURVES - TOP {top_n} MODELS")
    print("="*70)
    
    perf = pd.read_csv("../results/models/model_performance_summary.csv")
    perf = perf.sort_values('AUROC', ascending=False).head(top_n)
    top_model_names = perf['Model'].tolist()
    
    top_models_data = {k: v for k, v in models_data.items() if k in top_model_names}
    
    n_models = len(top_models_data)
    fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    
    if n_models == 1:
        axes = [axes]
    
    for idx, (model_name, data) in enumerate(top_models_data.items()):
        ax = axes[idx]
        y_true = data['y_true']
        y_pred_proba = data['y_pred_proba']
        
        prob_true, prob_pred = calibration_curve(
            y_true, y_pred_proba, n_bins=10, strategy='quantile'
        )
        
        brier = brier_score_loss(y_true, y_pred_proba)
        
        ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=2, alpha=0.7)
        ax.plot(prob_pred, prob_true, 's-', linewidth=2.5, markersize=8, color='#2E86AB')
        
        ax.set_xlabel('Predicted Probability', fontsize=12, fontweight='bold')
        ax.set_ylabel('Observed Probability', fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        
        ax.text(0.98, 0.02, f'Brier: {brier:.4f}',
               transform=ax.transAxes, fontsize=10,
               verticalalignment='bottom', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        print(f"  {model_name}: Brier = {brier:.4f}")
    
    plt.suptitle('Calibration Curves', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    plt.savefig(f'{output_dir}/calibration_curves.png', dpi=300, bbox_inches='tight')
    print(f"\n  Saved: {output_dir}/calibration_curves.png")
    plt.close()


In [6]:
def main():
    """Main analysis - matches paper methodology"""
    
    print("="*70)
    print("CLINICAL UTILITY ANALYSIS - EXACT PAPER REPLICATION")
    print("="*70)
    print("\nKey features:")
    print("  Savitzky-Golay smoothing (window=21, poly=3)")
    print("  Fine threshold steps (0.005)")
    print("  Min benefit clipping (-0.2)")
    print("  All models included")
    print("="*70)
    
    output_dir = "../results/clinical_utility"
    os.makedirs(output_dir, exist_ok=True)
    
    perf = pd.read_csv("../results/models/model_performance_summary.csv")
    
    print(f"\nLoading ALL {len(perf)} models...")
    models_to_load = perf['Model'].tolist()
    
    models_data = {}
    
    for model_name in models_to_load:
        try:
            model, X_test, y_test, y_pred_proba = load_model_and_data(model_name)
            
            models_data[model_name] = {
                'model': model,
                'X_test': X_test,
                'y_true': y_test,
                'y_pred_proba': y_pred_proba
            }
            
        except Exception as e:
            print(f"  ✗ Error loading {model_name}: {e}")
            continue
    
    if len(models_data) == 0:
        print("\nERROR: No models loaded!")
        return
    
    print(f"\nSuccessfully loaded {len(models_data)} models")
    
    # Create decision curve with paper's exact methodology
    create_decision_curve_PAPER_STYLE(models_data, output_dir)
    
    # Create calibration curves
    create_calibration_curves(models_data, output_dir, top_n=3)
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE - PAPER STYLE")
    print("="*70)
    print(f"\nResults in: {output_dir}/")
    print("  - decision_curve_analysis_all_models.png (with smoothing!)")
    print("  - decision_curve_analysis_all_models.svg")
    print("  - calibration_curves.png")
    print("="*70)


In [7]:
if __name__ == "__main__":
    main()

CLINICAL UTILITY ANALYSIS - EXACT PAPER REPLICATION

Key features:
  Savitzky-Golay smoothing (window=21, poly=3)
  Fine threshold steps (0.005)
  Min benefit clipping (-0.2)
  All models included

Loading ALL 10 models...

Loading LightGBM...
  Trained and calibrated

Loading XGBoost...
  Trained and calibrated

Loading Gradient Boosting...
  Trained and calibrated

Loading Random Forest...
  Trained and calibrated

Loading MLP...
  Using 30% training sample for speed
  Trained and calibrated

Loading AdaBoost...
  Using 30% training sample for speed
  Trained and calibrated

Loading Logistic Regression...
  Trained and calibrated

Loading SVM...
  Using 30% training sample for speed
  Trained and calibrated

Loading Gaussian Naive Bayes...
  ✗ Error loading Gaussian Naive Bayes: Model Gaussian Naive Bayes not supported

Loading Complement Naive Bayes...
  ✗ Error loading Complement Naive Bayes: Model Complement Naive Bayes not supported

Successfully loaded 8 models

DECISION CURVE A